## EDA with DuckDB

In [3]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd

### Defile the paths
The deafult option is set to "Books" category

In [4]:
DATA_DIR = Path("data")
CATEGORY = "Books"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"

In [5]:
print(REVIEWS_URL)

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Books.jsonl.gz


### Initialize an in-memory DB connection

In [6]:
c2 = duckdb.connect()

### Lets open the file online and see the first few lines

In [7]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Not a watercolor book! Seems like copies imo.,It is definitely not a watercolor book. The p...,[{'small_image_url': 'https://m.media-amazon.c...,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1642399598485,0,True
1,5.0,Updated: after 1st arrived damaged this one is...,Updated: after first book arrived very damaged...,[],0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640629604904,1,True
2,5.0,Excellent! I love it!,I bought it for the bag on the front so it pai...,[],1782490671,1782490671,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640383495102,0,True
3,5.0,Updated after 1st arrived damaged. Excellent,Updated: after 1st arrived damaged the replace...,[],0593138228,0593138228,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640364906602,0,False
4,5.0,Beautiful patterns!,I love this book! The patterns are lovely. I ...,[{'small_image_url': 'https://m.media-amazon.c...,0823098079,0823098079,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1637312253230,0,True


In [8]:
# executes right over the internet -- i just want five rows to preview so doesn't take long
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Books,Chaucer,"Hardcover – Import, January 1, 2004",{'avatar': 'https://m.media-amazon.com/images/...,4.5,29,[],[],8.23,[{'large': 'https://m.media-amazon.com/images/...,[],Peter Ackroyd (Author),"[Books, Literature & Fiction, History & Critic...","{'Publisher': '""Chatto & Windus; First Edition...",0701169850,None
1,Books,Notes from a Kidwatcher,First Edition,{'avatar': 'https://m.media-amazon.com/images/...,5.0,1,[Contains 23 selected articles by this influen...,"[About the Author, SANDRA WILDE, Ph.D., is wid...",3.52,[{'large': 'https://m.media-amazon.com/images/...,[],Sandra Wilde (Editor),"[Books, Reference, Words, Language & Grammar]","{'Publisher': '""Heinemann; First Edition (May ...",0435088688,None
2,Books,Service: A Navy SEAL at War,"Hardcover – May 8, 2012",{'avatar': 'https://m.media-amazon.com/images/...,4.7,3421,"[Marcus Luttrell, author of the #1 bestseller,...","[Review, Praise for SERVICE""An action-packed.....",17.17,[{'large': 'https://m.media-amazon.com/images/...,[],"Marcus Luttrell (Author), James D. Hornfischer","[Books, Biographies & Memoirs, Leaders & Notab...","{'Publisher': '""Little, Brown and Company; 1st...",0316185361,None
3,Books,Monstrous Stories #4: The Day the Mice Stood S...,"Paperback – October 29, 2013",<NA>,4.4,40,"[Funny, light-hearted monster stories that are...",[],7.43,[{'large': 'https://m.media-amazon.com/images/...,[],Dr. Roach (Author),"[Books, Children's Books, Science Fiction & Fa...","{'Publisher': '""Scholastic Paperbacks; Reprint...",0545425573,None
4,Buy a Kindle,Parker & Knight,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.5,381,"[From REMINGTON KANE, the author of The Taken!...",[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Remington Kane (Author) Format: Kindle Edition,"[Books, Mystery, Thriller & Suspense, Thriller...","{'Publication date': '""May 18, 2014""', 'Langua...",B00KFOP3RG,None


### Download and convert to parquet on the fly

So lets download and covert to parquet on the fly (remember, we pay conversion tax once and reuse for all consequent queries -- whether in duckdb or pandas ...). 

Even if we are performing parallel processing, we need full download here, so it will take time, depending on your connection + conversion overhead.

- Downloading only 20k (`LIMIT 20000`) for a quick inspection. 
- It take a few seconds to download (will depend on your network connection)  

In [20]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO 'data/raw/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [21]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO 'data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

### Merging the two files by joining on `parent_asin`

In [22]:
c2.execute("""
    COPY (
        SELECT r.*, m.title AS product_title, m.price,
                    m.average_rating, m.main_category, m.store
        FROM read_parquet('data/raw/reviews_raw.parquet') r
        LEFT JOIN read_parquet('data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO 'data/raw/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

### See it as a dataframe again

In [24]:
c2.execute(f"SELECT * FROM read_parquet('data/raw/merged.parquet')").df()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store
0,5.0,Still good after all these years,My grandson just turned 3 and is really loving...,[],0394800184,0394800184,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1396297656000,0,True,Are You My Mother ?,5.29,4.9,Books,P.D. Eastman (Author)
1,5.0,A great reminder about forgiveness,"As someone who experienced needing to forgive,...",[],0718039874,0718039874,AG5Y7PRM4OD7TC23Q6S7KMNL2XNQ,1631718402452,0,True,Forgiving What You Can't Forget: Discover How ...,17.96,4.8,Books,Lysa TerKeurst (Author)
2,5.0,Very cute story!,My daughter loved this series! She’s very into...,[],1681196522,1681196522,AGFJAPF5SAJG4AGK3LXKAG7IT3PQ,1530720280488,0,True,Unicorn Princesses 6: Moon's Dance,6.99,4.8,Books,"Emily Bliss (Author), Sydney Hanson (Illustra..."
3,5.0,Very Good Read,I am reading this now. It is very funny and i...,[],1951627997,1951627997,AFJHM4DEMY7IU6ZCUBCETLREE43Q,1635451281488,0,True,Apropos of Nothing: Autobiography,17.99,4.5,Books,Woody Allen (Author)
4,4.0,Meh,I have an iphone and needed to learn to take b...,[],1119687799,1119687799,AFFZVSTUS3U2ZD22A2NPZSKOCPGQ,1603314722598,2,False,iPhone Photography For Dummies,17.89,4.5,Books,Mark Hemmings (Author)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,5.0,Love Brittney and this book,Love Brittney and this book. There are a coup...,[],1250112575,1250112575,AENZL473YXQW64RJJASTJBH4Y53A,1531667791398,0,True,None,NaN,NaN,None,None
19996,5.0,Timeless assessment of the USA,Although the book was written about 100-years ...,[],0486408906,0486408906,AENZL473YXQW64RJJASTJBH4Y53A,1483640881000,7,True,None,NaN,NaN,None,None
19997,4.0,An excellent timely topic as we witness a voca...,A thought provoking book! Dr. Cone makes the c...,[],1626980055,1626980055,AENZL473YXQW64RJJASTJBH4Y53A,1480878793000,5,True,None,NaN,NaN,None,None
19998,5.0,Good history of North Carolina politics,I saw Rev Barber on The Rachel Maddow Show ; I...,[],0807083607,0807083607,AENZL473YXQW64RJJASTJBH4Y53A,1463232430000,9,True,None,NaN,NaN,None,None


### Rating distribution

In [26]:
# EDA example duckdb Rating distribution
c2.execute("""
    SELECT
        rating,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('data/raw/merged.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()

,rating,cnt,pct
0,1.0,459,2.30
1,2.0,719,3.60
2,3.0,1923,9.62
3,4.0,4499,22.50
4,5.0,12400,62.00


### Stratified sample: 

**IF resulting parquet might still be too large, but you don't just want to take the first 20000-30000 rows**, you want to sample reviews across main dimensions: long and short reviews, different stars,

What queries do: 

(1) `labelled`
- Joins reviews with metadata on `parent_asin`.
- Adds two labels to each review:
- `rating_bucket`: based on the product’s average rating
- `len_tier`: based on review length
Filters out:
- Reviews with missing/empty text
- Products without an average rating

This is the base dataset everything else builds on.

(2) `one_per_product` $\to$ `ranked`

`one_per_product`

- Within each stratum (rating_bucket x len_tier x `verified_purchase`):
- bRank reviews per product by helpful_vote DESC
- We want to keep only the best review per product per cell
(e.g., from 500 reviews --> 1 candidate)

- Take those top reviews (`product_rank = 1`)
-Rank them across the entire cell by: `helpful_vote DESC`, `random()`

This decides which products make the final cut when more than SAMPLE_PER_STRATUM products exist in a cell.

(3) Final `SELECT` Keeps only the top `SAMPLE_PER_STRATUM` rows per cell (via `stratum_rank <= N`), drops the internal ranking columns, and writes to Parquet. The result is a balanced dataset with at most N rows per stratum cell, each row from a different product, prioritising reviews with the most helpful votes.

In [22]:
SAMPLE_PER_STRATUM = 50    # reviews to keep per stratum cell (floor guarantee)
MIN_TEXT_LEN       = 20    # drop near-empty reviews (chars)
SHORT_MAX          = 100   # short: text < SHORT_MAX chars
MEDIUM_MAX         = 500   # medium: SHORT_MAX ≤ text < MEDIUM_MAX chars
                        # long: text ≥ MEDIUM_MAX chars

OUTPUT = 'data/stratified_sample.parquet'

c2.execute(f"""
COPY (
WITH labelled AS (
    SELECT
        text, rating, verified_purchase, helpful_vote,
        parent_asin, user_id, timestamp,
        product_title, price,
        average_rating, main_category,

        CASE
            WHEN average_rating >= 4.6 THEN '4.6-5.0'
            WHEN average_rating >= 4.4 THEN '4.4-4.5'
            WHEN average_rating >= 4.1 THEN '4.1-4.3'
            WHEN average_rating >= 3.7 THEN '3.7-4.0'
            WHEN average_rating >= 3.1 THEN '3.1-3.6'
            ELSE                              '<=3.0'
        END AS rating_bucket,

        CASE
            WHEN LENGTH(text) < {SHORT_MAX}  THEN 'short'
            WHEN LENGTH(text) < {MEDIUM_MAX} THEN 'medium'
            ELSE                                    'long'
        END AS len_tier

    FROM read_parquet('data/merged.parquet')
    WHERE text IS NOT NULL
    AND LENGTH(text) >= {MIN_TEXT_LEN}
    AND average_rating IS NOT NULL
),

-- Step 1: one review per product per cell: prevents a single popular product flooding a stratum, you can redefine
one_per_product AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase, parent_asin
            ORDER BY helpful_vote DESC, random()
        ) AS product_rank
    FROM labelled
),

-- Step 2: rank within each stratum cell, helpful reviews first
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY rating_bucket, len_tier, verified_purchase
            ORDER BY helpful_vote DESC, random()
        ) AS stratum_rank
    FROM one_per_product
    WHERE product_rank = 1
)

SELECT * EXCLUDE (product_rank, stratum_rank)
FROM ranked
WHERE stratum_rank <= {SAMPLE_PER_STRATUM}
)
TO '{OUTPUT}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")